In [1]:
# Cell 01 | Inspect ToyAIKit agent abstractions
# Purpose: Verify the main ToyAIKit components and inspect the framework interfaces.
# Key points: Compare the framework abstractions with the explicit agent loop
#             implemented in Lesson 14 before using them to run an agent.
# Execution: This cell only imports and inspects Python objects.
#            No OpenAI API request is made.

import inspect

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner


print("ToyAIKit components imported successfully.")

print("\nCore abstractions:")
print("Tools:", Tools)
print("OpenAIClient:", OpenAIClient)
print("OpenAIResponsesRunner:", OpenAIResponsesRunner)

print("\nSelected method signatures:")
print("Tools.add_tool:", inspect.signature(Tools.add_tool))
print("OpenAIResponsesRunner.loop:", inspect.signature(OpenAIResponsesRunner.loop))# Cell 01 | Inspect ToyAIKit agent abstractions
# Purpose: Verify the main ToyAIKit components and inspect the framework interfaces.
# Key points: Compare the framework abstractions with the explicit agent loop
#             implemented in Lesson 14 before using them to run an agent.
# Execution: This cell only imports and inspects Python objects.
#            No OpenAI API request is made.

import inspect

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner


print("ToyAIKit components imported successfully.")

print("\nCore abstractions:")
print("Tools:", Tools)
print("OpenAIClient:", OpenAIClient)
print("OpenAIResponsesRunner:", OpenAIResponsesRunner)

print("\nSelected method signatures:")
print("Tools.add_tool:", inspect.signature(Tools.add_tool))
print("OpenAIResponsesRunner.loop:", inspect.signature(OpenAIResponsesRunner.loop))

ToyAIKit components imported successfully.

Core abstractions:
Tools: <class 'toyaikit.tools.Tools'>
OpenAIClient: <class 'toyaikit.llm.OpenAIClient'>
OpenAIResponsesRunner: <class 'toyaikit.chat.runners.OpenAIResponsesRunner'>

Selected method signatures:
Tools.add_tool: (self, function, schema=None)
OpenAIResponsesRunner.loop: (self, prompt: str, previous_messages: list[dict] = None, callback: toyaikit.chat.runners.RunnerCallback = None, output_format: pydantic.main.BaseModel = None) -> toyaikit.chat.runners.LoopResult
ToyAIKit components imported successfully.

Core abstractions:
Tools: <class 'toyaikit.tools.Tools'>
OpenAIClient: <class 'toyaikit.llm.OpenAIClient'>
OpenAIResponsesRunner: <class 'toyaikit.chat.runners.OpenAIResponsesRunner'>

Selected method signatures:
Tools.add_tool: (self, function, schema=None)
OpenAIResponsesRunner.loop: (self, prompt: str, previous_messages: list[dict] = None, callback: toyaikit.chat.runners.RunnerCallback = None, output_format: pydantic.main.

In [2]:
# Cell 02 | Register and inspect the search tool
# Purpose: Register the project's verified search() function with ToyAIKit Tools.
# Key points: Observe how ToyAIKit stores a Python function after registration
#             before relying on any framework-generated tool schema.
# Execution: Run after Cell 01 from the notebooks directory.
#            This cell performs local setup only; no OpenAI API request is made.

from dotenv import load_dotenv
from openai import OpenAI
from sqlitesearch import TextSearchIndex

from rag_helper import RAGBase


# Load the local project environment.
load_dotenv("../.env")

# Reuse the verified persistent SQLite retrieval configuration.
sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db",
)

client = OpenAI()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=client,
)


def search(query: str):
    """Search the LLM Zoomcamp FAQ for documents relevant to the query.

    Args:
        query: The search query used to retrieve relevant FAQ documents.

    Returns:
        A list of relevant FAQ documents.
    """
    return assistant.search(query)


# Create the ToyAIKit tool registry.
tools = Tools()

# Register the Python function directly.
tools.add_tool(search)


print("Registered function:", search.__name__)
print("Documents available:", sqlite_index.count())

print("\nPublic Tools attributes:")
public_attributes = [
    name
    for name in dir(tools)
    if not name.startswith("_")
]
print(public_attributes)

print("\nTools instance state keys:")
print(list(vars(tools).keys()))

print("\nTools instance state:")
for name, value in vars(tools).items():
    print(f"\n{name} ({type(value).__name__})")
    print(value)

Registered function: search
Documents available: 139

Public Tools attributes:
['add_tool', 'add_tools', 'function_call', 'functions', 'get_tools', 'tools']

Tools instance state keys:
['tools', 'functions']

Tools instance state:

tools (dict)
{'search': {'type': 'function', 'name': 'search', 'description': 'Search the LLM Zoomcamp FAQ for documents relevant to the query.\n\nArgs:\n    query: The search query used to retrieve relevant FAQ documents.\n\nReturns:\n    A list of relevant FAQ documents.', 'parameters': {'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'query parameter'}}, 'required': ['query'], 'additionalProperties': False}}}

functions (dict)
{'search': <function search at 0x000001FD7A3438A0>}


In [3]:
# Cell 03 | Inspect the generated tool contract
# Purpose: Inspect the public tool schema and execution interface exposed by ToyAIKit.
# Key points: Compare the generated schema with the manual Function Calling contract
#             from Lesson 13 before allowing the framework to execute tools.
# Execution: This cell only inspects local Python objects.
#            No OpenAI API request is made.

import inspect
import json


generated_tools = tools.get_tools()

print("Generated tools:")
print(json.dumps(generated_tools, indent=2, ensure_ascii=False))

print("\nTool registry names:")
print(list(tools.tools.keys()))

print("\nFunction registry names:")
print(list(tools.functions.keys()))

print("\nExecution interfaces:")
print("Tools.get_tools:", inspect.signature(tools.get_tools))
print("Tools.function_call:", inspect.signature(tools.function_call))

Generated tools:
[
  {
    "type": "function",
    "name": "search",
    "description": "Search the LLM Zoomcamp FAQ for documents relevant to the query.\n\nArgs:\n    query: The search query used to retrieve relevant FAQ documents.\n\nReturns:\n    A list of relevant FAQ documents.",
    "parameters": {
      "type": "object",
      "properties": {
        "query": {
          "type": "string",
          "description": "query parameter"
        }
      },
      "required": [
        "query"
      ],
      "additionalProperties": false
    }
  }
]

Tool registry names:
['search']

Function registry names:
['search']

Execution interfaces:
Tools.get_tools: ()
Tools.function_call: (tool_call_response) -> openai.types.responses.response_input_param.FunctionCallOutput


In [4]:
# Cell 04 | Inspect the ToyAIKit tool dispatch implementation
# Purpose: Inspect how ToyAIKit converts an OpenAI function call into local Python execution.
# Key points: Identify function lookup, argument parsing, execution, and
#             FunctionCallOutput construction inside the framework.
# Execution: This cell only inspects installed Python source code.
#            No OpenAI API request or retrieval call is made.

import inspect


function_call_source = inspect.getsource(Tools.function_call)

print("Tools.function_call source:")
print(function_call_source)

Tools.function_call source:
    def function_call(self, tool_call_response) -> FunctionCallOutput:
        """
        Handle a function call from the LLM.

        Args:
            tool_call_response: The tool call response from the LLM.

        Returns:
            dict: The result of the function call or error details if the call fails.
        """
        try:
            function_name = tool_call_response.name
            arguments_raw = tool_call_response.arguments
            arguments = json.loads(arguments_raw)

            call_id = tool_call_response.call_id

            if function_name not in self.functions:
                raise KeyError(f"Unknown function: {function_name}")

            f = self.functions[function_name]
            result = f(**arguments)

            return FunctionCallOutput(
                type="function_call_output",
                call_id=call_id,
                output=json.dumps(result, indent=2),
            )

        except Exception as e:


In [5]:
# Cell 05 | Verify local tool dispatch
# Purpose: Verify that ToyAIKit can dispatch an LLM-style tool request
#          to the registered Python search() function.
# Key points: Simulate a function call locally without using the OpenAI API.
# Execution: Run after Cell 01–04.
#            This cell performs one real FAQ retrieval but no LLM request.

import json
from types import SimpleNamespace


# Simulate the structure of a function call requested by an LLM.
fake_tool_call = SimpleNamespace(
    name="search",
    arguments=json.dumps(
        {
            "query": "Can I still join the course after it has started?"
        }
    ),
    call_id="local_test_call_001",
)

# Let ToyAIKit:
# 1. read the requested function name,
# 2. parse the JSON arguments,
# 3. execute the registered Python function,
# 4. package the result as function_call_output.
tool_output = tools.function_call(fake_tool_call)

print("Tool output type:", type(tool_output).__name__)
print("Tool output:")
print(tool_output)


# Decode the returned search result for a simple verification.
retrieved_documents = json.loads(tool_output["output"])

print("\nRetrieved documents:", len(retrieved_documents))

if retrieved_documents:
    print("Top result question:")
    print(retrieved_documents[0]["question"])

Tool output type: dict
Tool output:
{'type': 'function_call_output', 'call_id': 'local_test_call_001', 'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "a9353fadfe",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "The homework submission form is still open even though the deadline has passed \\u2014 can I still submit?",\n    "answer": "Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed \\u2014 so while it\'s still open, go ahead and submit."\n  },\n  {\n    "id": "9f689c185f",\n    "course": "

In [6]:
# Cell 06 | Inspect ToyAIKit runner construction
# Purpose: Inspect the constructors required to build a ToyAIKit agent runner.
# Key points: Confirm the installed API before creating the real agent.
# Execution: Local inspection only. No OpenAI API request is made.

import inspect


print("OpenAIClient constructor:")
print(inspect.signature(OpenAIClient))

print("\nOpenAIResponsesRunner constructor:")
print(inspect.signature(OpenAIResponsesRunner))

OpenAIClient constructor:
(model: str = 'gpt-4o-mini', client: openai.OpenAI = None, extra_kwargs: dict = None)

OpenAIResponsesRunner constructor:
(tools: toyaikit.tools.Tools = None, developer_prompt: str = "You're a helpful assistant.", chat_interface: toyaikit.chat.interface.ChatInterface = None, llm_client: toyaikit.llm.LLMClient = None, pricing_config: toyaikit.pricing.PricingConfig = None)


In [7]:
# Cell 07 | Build the ToyAIKit agent runner
# Purpose: Assemble the verified search tool, OpenAI client, and agent instructions
#          into a ToyAIKit OpenAIResponsesRunner.
# Key points: This creates the framework objects only.
#             It does not run the agent or make an API request.
# Execution: Run after Cell 01–06.

# Wrap the existing OpenAI SDK client with ToyAIKit's LLM client.
llm_client = OpenAIClient(
    model="gpt-5.6",
    client=client,
)

# Instructions that define how the RAG agent should behave.
developer_prompt = """
You are a RAG assistant for the LLM Zoomcamp FAQ.

Use the search tool when factual course information is needed.
Base factual answers on retrieved FAQ evidence.
If the available evidence is insufficient, search again when useful.
Stop using tools once there is enough evidence to answer.
If the evidence does not support an answer, say that the available FAQ
does not contain enough information.
""".strip()

# Build the ToyAIKit agent runner.
runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=developer_prompt,
    llm_client=llm_client,
)

print("ToyAIKit runner created successfully.")
print("LLM client type:", type(llm_client).__name__)
print("Runner type:", type(runner).__name__)
print("Registered tools:", list(tools.functions.keys()))

ToyAIKit runner created successfully.
LLM client type: OpenAIClient
Runner type: OpenAIResponsesRunner
Registered tools: ['search']


In [8]:
# Cell 08 | Run the ToyAIKit agent loop
# Purpose: Execute the same RAG task through ToyAIKit's framework-managed loop.
# Key points: Let the framework handle model decisions, tool execution,
#             tool outputs, history updates, and stopping.
# Execution: This cell makes real OpenAI API requests and may execute FAQ search.

user_question = (
    "Can I still join the LLM Zoomcamp course after it has started?"
)

result = runner.loop(
    prompt=user_question,
)

print("ToyAIKit agent completed.")
print("Result type:", type(result).__name__)

print("\nResult attributes:")
print(vars(result))

ToyAIKit agent completed.
Result type: LoopResult

Result attributes:
{'new_messages': [EasyInputMessage(content='You are a RAG assistant for the LLM Zoomcamp FAQ.\n\nUse the search tool when factual course information is needed.\nBase factual answers on retrieved FAQ evidence.\nIf the available evidence is insufficient, search again when useful.\nStop using tools once there is enough evidence to answer.\nIf the evidence does not support an answer, say that the available FAQ\ndoes not contain enough information.', role='developer', phase=None, type=None), EasyInputMessage(content='Can I still join the LLM Zoomcamp course after it has started?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"Can I still join the LLM Zoomcamp course after it has started late enrollment?"}', call_id='call_BVGDUrB72jfnAPDZDBWaGQDx', name='search', type='function_call', id='fc_0ac2b53a72ec02a3006a7c919f84b48198a9435b07ca4bc30c', caller=None, namespace=None, status='comple

In [9]:
# Cell 09 | Audit the ToyAIKit agent trajectory
# Purpose: Inspect the framework-managed trajectory, final answer, and usage.
# Key points: Verify tool usage and identify whether the final claims are
#             actually supported by the retrieved FAQ evidence.
# Execution: Uses the existing Cell 08 result only.
#            No additional OpenAI API request is made.

import json


function_calls = []
function_outputs = []
final_messages = []

for message in result.new_messages:
    message_type = (
        message.get("type")
        if isinstance(message, dict)
        else getattr(message, "type", None)
    )

    if message_type == "function_call":
        function_calls.append(message)

    elif message_type == "function_call_output":
        function_outputs.append(message)

    elif message_type == "message":
        final_messages.append(message)


print("=== ToyAIKit trajectory audit ===")
print("New messages:", len(result.new_messages))
print("Function calls:", len(function_calls))
print("Function outputs:", len(function_outputs))
print("Final messages:", len(final_messages))

print("\n=== Final answer ===")
print(result.last_message)

print("\n=== Usage ===")
print("Input tokens:", result.tokens.input_tokens)
print("Output tokens:", result.tokens.output_tokens)
print("Total cost:", result.cost.total_cost)


if function_outputs:
    retrieved_documents = json.loads(function_outputs[0]["output"])

    print("\n=== Retrieved evidence ===")

    for index, document in enumerate(retrieved_documents, start=1):
        print(f"\nEvidence {index}")
        print("Question:", document["question"])
        print("Answer:", document["answer"])

=== ToyAIKit trajectory audit ===
New messages: 5
Function calls: 1
Function outputs: 1
Final messages: 1

=== Final answer ===
Yes, you can still join after the course has started. You can begin learning and submitting homework while the submission forms are open.

To receive a certificate, make sure you submit the required project before project submissions close. Check the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/) for current deadlines.

=== Usage ===
Input tokens: 965
Output tokens: 109
Total cost: 0.008095

=== Retrieved evidence ===

Evidence 1
Question: I just discovered the course. Can I still join?
Answer: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

Evidence 2
Question: Where can I track the LLM Zoomcamp syllabus, deadlines, homework, and progress?
Answer: Use the [LLM Zoomcamp course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

It con

In [10]:
# Cell 10 | Harden grounding instructions and rerun the agent
# Purpose: Reduce unsupported claims by requiring every factual statement
#          to be explicitly supported by retrieved FAQ evidence.
# Key points: Keep retrieval unchanged and improve only the agent instructions.
# Execution: Makes real OpenAI API requests and may execute the search tool.

grounded_developer_prompt = """
You are a RAG assistant for the LLM Zoomcamp FAQ.

Use the search tool whenever factual course information is needed.

Grounding rules:
- Base every factual claim on retrieved FAQ evidence.
- Do not add requirements, deadlines, policies, or conditions unless they are
  explicitly stated in the retrieved evidence.
- Do not turn optional or recommended actions into mandatory requirements.
- Keep different topics separate, such as course participation, homework,
  project submission, and certificate requirements.
- If the evidence only supports part of the answer, answer only that supported part.
- If evidence is insufficient or conflicting, search again when useful.
- If sufficient evidence still cannot be found, clearly say that the available
  FAQ does not contain enough information.

Stop using tools once there is enough evidence to answer.
""".strip()


grounded_runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=grounded_developer_prompt,
    llm_client=llm_client,
)

grounded_result = grounded_runner.loop(
    prompt=user_question,
)

print("=== Grounded ToyAIKit rerun ===")
print(grounded_result.last_message)

print("\n=== Usage ===")
print("Input tokens:", grounded_result.tokens.input_tokens)
print("Output tokens:", grounded_result.tokens.output_tokens)
print("Total cost:", grounded_result.cost.total_cost)

=== Grounded ToyAIKit rerun ===
Yes, you can still join after the course has started.

If you want a certificate, you must submit your project while project submissions are still being accepted. Check the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/) for current deadlines and progress tracking.

=== Usage ===
Input tokens: 1130
Output tokens: 99
Total cost: 0.00862


In [11]:
# Cell 11 | Compare the explicit agent loop with ToyAIKit
# Purpose: Summarize which responsibilities remain visible in the hand-built
#          agent loop and which responsibilities ToyAIKit abstracts away.
# Key points: Framework convenience reduces boilerplate, but retrieval,
#             tool permissions, prompts, and evaluation remain application concerns.
# Execution: Documentation-only output. No API or retrieval calls are made.

comparison = {
    "Tool schema": {
        "Explicit loop": "Manually defined JSON schema",
        "ToyAIKit": "Generated from Python function metadata",
    },
    "Tool registry": {
        "Explicit loop": "Manual dispatch logic",
        "ToyAIKit": "Tools() manages schemas and Python functions",
    },
    "Argument parsing": {
        "Explicit loop": "json.loads(call.arguments)",
        "ToyAIKit": "Handled by Tools.function_call()",
    },
    "Tool execution": {
        "Explicit loop": "Application dispatches search(**arguments)",
        "ToyAIKit": "Tools.function_call() dispatches the registered function",
    },
    "Function call output": {
        "Explicit loop": "Built manually",
        "ToyAIKit": "Built by Tools.function_call()",
    },
    "Agent history": {
        "Explicit loop": "Managed manually",
        "ToyAIKit": "Managed by OpenAIResponsesRunner.loop()",
    },
    "Repeated decisions": {
        "Explicit loop": "Implemented in run_agent()",
        "ToyAIKit": "Managed by runner.loop()",
    },
    "Final answer": {
        "Explicit loop": "Extracted manually",
        "ToyAIKit": "Available as LoopResult.last_message",
    },
    "Token and cost tracking": {
        "Explicit loop": "Not built into the lesson baseline",
        "ToyAIKit": "Available in LoopResult.tokens and LoopResult.cost",
    },
}

print("=== Explicit Agent Loop vs ToyAIKit ===")

for responsibility, implementation in comparison.items():
    print(f"\n{responsibility}")
    print("  Explicit loop:", implementation["Explicit loop"])
    print("  ToyAIKit:     ", implementation["ToyAIKit"])

print("\n=== Verified design boundary ===")
print("ToyAIKit abstracts agent-loop boilerplate.")
print("The application still owns retrieval behavior, registered tools,")
print("developer instructions, permissions, grounding checks, and evaluation.")

=== Explicit Agent Loop vs ToyAIKit ===

Tool schema
  Explicit loop: Manually defined JSON schema
  ToyAIKit:      Generated from Python function metadata

Tool registry
  Explicit loop: Manual dispatch logic
  ToyAIKit:      Tools() manages schemas and Python functions

Argument parsing
  Explicit loop: json.loads(call.arguments)
  ToyAIKit:      Handled by Tools.function_call()

Tool execution
  Explicit loop: Application dispatches search(**arguments)
  ToyAIKit:      Tools.function_call() dispatches the registered function

Function call output
  Explicit loop: Built manually
  ToyAIKit:      Built by Tools.function_call()

Agent history
  Explicit loop: Managed manually
  ToyAIKit:      Managed by OpenAIResponsesRunner.loop()

Repeated decisions
  Explicit loop: Implemented in run_agent()
  ToyAIKit:      Managed by runner.loop()

Final answer
  Explicit loop: Extracted manually
  ToyAIKit:      Available as LoopResult.last_message

Token and cost tracking
  Explicit loop: Not bu